# Sesión 2: Consultas de Selección con Tablas Relacionadas
## Clase: INNER JOIN y LEFT JOIN en SQL

**Módulo:** Fundamentos de Programación Python para el Análisis de Datos

**Contenido:**
- INNER JOIN: Unión de registros coincidentes
- LEFT JOIN: Conservación de todos los registros de una tabla
- Comparación y buenas prácticas
- Actividades prácticas integradas

## Configuración Inicial

Instalamos las librerías necesarias y importamos módulos

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import os

# Verificar disponibilidad de librerías
print("Librerías importadas correctamente:")
print(f"- sqlite3: {sqlite3.version}")
print(f"- pandas: {pd.__version__}")
print(f"- numpy: {np.__version__}")

## SLIDE 3: Desafío Inicial

**Contexto:** Trabajas como analista de datos en una empresa de servicios digitales que gestiona contratos con distintos clientes y registra las órdenes de servicio. Necesitas combinar información de clientes con sus órdenes correspondientes para generar reportes.

**Problema:** ¿Cómo obtener información integrada de clientes y sus órdenes en una sola consulta?

In [ ]:
# Crear base de datos en memoria
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

# Verificar que la conexión se estableció
print("Conexión a base de datos SQLite establecida correctamente")
print(f"Versión de SQLite: {sqlite3.version}")

## SLIDE 5-6: INNER JOIN - Concepto y Sintaxis Básica

**¿Qué es INNER JOIN?**
INNER JOIN permite combinar registros de dos o más tablas siempre que exista una correspondencia basada en una condición (clave de relación).

**Sintaxis básica:**
```sql
SELECT c.nombre, o.fecha_orden
FROM clientes c
INNER JOIN ordenes o ON c.id_cliente = o.id_cliente;
```

In [ ]:
# Crear tabla CLIENTES
cursor.execute('''
CREATE TABLE clientes (
    id_cliente INTEGER PRIMARY KEY,
    nombre TEXT NOT NULL,
    email TEXT,
    ciudad TEXT,
    fecha_registro DATE
)
''')

# Insertar datos de clientes
clientes_data = [
    (1, 'Juan García', 'juan.garcia@email.com', 'Madrid', '2023-01-15'),
    (2, 'María López', 'maria.lopez@email.com', 'Barcelona', '2023-02-20'),
    (3, 'Carlos Martín', 'carlos.martin@email.com', 'Valencia', '2023-03-10'),
    (4, 'Ana Rodríguez', 'ana.rodriguez@email.com', 'Bilbao', '2023-04-05'),
    (5, 'Pedro Sánchez', 'pedro.sanchez@email.com', 'Madrid', '2023-05-12')
]

cursor.executemany(
    'INSERT INTO clientes VALUES (?, ?, ?, ?, ?)',
    clientes_data
)

conn.commit()
print("Tabla 'clientes' creada e insertados 5 registros")

# Verificar contenido
df_clientes = pd.read_sql('SELECT * FROM clientes', conn)
print("\nContenido de la tabla clientes:")
print(df_clientes)

In [ ]:
# Crear tabla ORDENES
cursor.execute('''
CREATE TABLE ordenes (
    id_orden INTEGER PRIMARY KEY,
    id_cliente INTEGER NOT NULL,
    fecha_orden DATE NOT NULL,
    monto DECIMAL(10, 2),
    estado TEXT,
    FOREIGN KEY(id_cliente) REFERENCES clientes(id_cliente)
)
''')

# Insertar datos de órdenes
# Nota: El cliente 5 no tiene órdenes (se usará para demostrar LEFT JOIN)
ordenes_data = [
    (1, 1, '2024-01-10', 150.00, 'completada'),
    (2, 1, '2024-02-15', 200.00, 'completada'),
    (3, 2, '2024-01-20', 300.00, 'completada'),
    (4, 3, '2024-02-05', 175.50, 'pendiente'),
    (5, 3, '2024-03-10', 250.00, 'completada'),
    (6, 4, '2024-02-28', 125.75, 'cancelada'),
    (7, 2, '2024-03-15', 400.00, 'pendiente')
]

cursor.executemany(
    'INSERT INTO ordenes VALUES (?, ?, ?, ?, ?)',
    ordenes_data
)

conn.commit()
print("Tabla 'ordenes' creada e insertados 7 registros")
print("Nota: El cliente 5 (Pedro Sánchez) NO tiene órdenes registradas\n")

# Verificar contenido
df_ordenes = pd.read_sql('SELECT * FROM ordenes', conn)
print("Contenido de la tabla ordenes:")
print(df_ordenes)

## SLIDE 7: Aplicación Práctica de INNER JOIN

**Uso:** Este tipo de combinación se usa cuando sólo se requiere la información de registros que tienen correspondencia en ambas tablas.

**Ejemplo práctico:** Obtener nombre de clientes junto a sus órdenes

In [ ]:
# INNER JOIN: Solo clientes con órdenes
query_inner_join = '''
SELECT 
    c.id_cliente,
    c.nombre,
    c.ciudad,
    o.id_orden,
    o.fecha_orden,
    o.monto,
    o.estado
FROM clientes c
INNER JOIN ordenes o ON c.id_cliente = o.id_cliente
ORDER BY c.id_cliente, o.fecha_orden
'''

df_inner = pd.read_sql(query_inner_join, conn)
print("RESULTADO INNER JOIN:")
print(f"Total de registros: {len(df_inner)}")
print("\nDatos:")
print(df_inner.to_string())
print("\n** Nota: El cliente 5 (Pedro Sánchez) NO aparece porque no tiene órdenes **")

## SLIDE 8: Buenas Prácticas en INNER JOIN

**Buenas prácticas:**
1. Usar alias para mejorar la legibilidad (c, o)
2. Siempre identificar las llaves de relación con claridad
3. Especificar columnas explícitamente en SELECT

**Errores comunes:**
1. Olvidar la condición ON (producto cartesiano)
2. Usar condiciones de filtro incorrectas
3. No usar alias cuando hay nombres ambiguos

In [ ]:
# EJEMPLO CORRECTO: Con alias y condición clara
print("✓ FORMA CORRECTA - Usando alias y especificando columnas:\n")
query_correcta = '''
SELECT 
    c.nombre,
    o.fecha_orden,
    o.monto
FROM clientes c
INNER JOIN ordenes o ON c.id_cliente = o.id_cliente
'''
print(query_correcta)
df_correcta = pd.read_sql(query_correcta, conn)
print(df_correcta)

In [ ]:
# EJEMPLO DE ERROR COMÚN 1: Sin condición ON (comentado para no ejecutar)
print("✗ ERROR COMÚN 1 - Sin condición ON (genera producto cartesiano):\n")
query_error1 = '''
SELECT c.nombre, o.fecha_orden
FROM clientes c, ordenes o
-- Esto generaría 5 * 7 = 35 registros sin sentido
'''
print(query_error1)
print("⚠ No ejecutado porque generaría datos incorrectos")

print("\n✗ ERROR COMÚN 2 - No usar alias (ambigüedad):\n")
query_error2 = '''
SELECT nombre, fecha_orden  -- ¿De qué tabla?
FROM clientes
INNER JOIN ordenes ON clientes.id_cliente = ordenes.id_cliente
'''
print(query_error2)

## SLIDE 9-10: LEFT JOIN - Concepto y Sintaxis Básica

**¿Qué es LEFT JOIN?**
LEFT JOIN (o combinación externa izquierda) permite unir dos tablas manteniendo TODOS los registros de la tabla izquierda, incluidos aquellos sin correspondencia en la tabla derecha (que aparecen como NULL).

**Sintaxis básica:**
```sql
SELECT c.nombre, o.fecha_orden
FROM clientes c
LEFT JOIN ordenes o ON c.id_cliente = o.id_cliente;
```

In [ ]:
# LEFT JOIN: Todos los clientes, con o sin órdenes
query_left_join = '''
SELECT 
    c.id_cliente,
    c.nombre,
    c.ciudad,
    o.id_orden,
    o.fecha_orden,
    o.monto,
    o.estado
FROM clientes c
LEFT JOIN ordenes o ON c.id_cliente = o.id_cliente
ORDER BY c.id_cliente, o.fecha_orden
'''

df_left = pd.read_sql(query_left_join, conn)
print("RESULTADO LEFT JOIN:")
print(f"Total de registros: {len(df_left)}")
print("\nDatos:")
print(df_left.to_string())
print("\n** Nota: El cliente 5 (Pedro Sánchez) APARECE porque es un LEFT JOIN, con NULLs en las columnas de órdenes **")

## SLIDE 11: Aplicación Práctica de LEFT JOIN

**Uso:** Se utiliza cuando es importante mantener todos los registros de la tabla principal, incluso si no tienen correspondencia en la otra tabla.

**Casos de uso:**
- Identificar clientes sin compras
- Detectar registros huérfanos
- Análisis de cobertura de datos

In [ ]:
# Análisis: Identificar clientes sin órdenes
query_sin_ordenes = '''
SELECT 
    c.id_cliente,
    c.nombre,
    c.email,
    COUNT(o.id_orden) as total_ordenes
FROM clientes c
LEFT JOIN ordenes o ON c.id_cliente = o.id_cliente
GROUP BY c.id_cliente, c.nombre, c.email
ORDER BY total_ordenes ASC
'''

df_sin_ordenes = pd.read_sql(query_sin_ordenes, conn)
print("Análisis: Clientes por número de órdenes")
print(df_sin_ordenes.to_string())
print("\nClientes sin órdenes (activos pero sin ventas):")
print(df_sin_ordenes[df_sin_ordenes['total_ordenes'] == 0])

## SLIDE 12: Buenas Prácticas en LEFT JOIN

**Buenas prácticas:**
1. Revisar cuántos registros NULL aparecen en los resultados
2. Usar COALESCE() para manejar valores NULL
3. Identificar claramente la tabla "izquierda" (la que se preserva)

**Errores comunes:**
1. Confundir cuál es la tabla izquierda y derecha
2. Agregar condiciones de filtro que conviertan LEFT JOIN en INNER JOIN
3. No manejar correctamente los valores NULL

In [ ]:
# BUENA PRÁCTICA: Manejar NULLs con COALESCE
print("✓ BUENA PRÁCTICA - Usando COALESCE para manejar NULL:\n")
query_coalesce = '''
SELECT 
    c.nombre,
    c.ciudad,
    COALESCE(o.fecha_orden, 'Sin órdenes') as fecha_orden,
    COALESCE(CAST(o.monto AS TEXT), 'N/A') as monto
FROM clientes c
LEFT JOIN ordenes o ON c.id_cliente = o.id_cliente
ORDER BY c.id_cliente
'''
df_coalesce = pd.read_sql(query_coalesce, conn)
print(df_coalesce.to_string())

In [ ]:
# ERROR COMÚN: Agregar condición WHERE que convierte LEFT JOIN en INNER JOIN
print("\n✗ ERROR COMÚN - Condición WHERE que convierte LEFT a INNER:\n")
query_error_where = '''
SELECT c.nombre, o.fecha_orden, o.monto
FROM clientes c
LEFT JOIN ordenes o ON c.id_cliente = o.id_cliente
WHERE o.monto > 200  -- ¡Esto elimina registros NULL! Ahora es INNER JOIN
'''
print("Query (INCORRECTA):")
print(query_error_where)
df_error = pd.read_sql(query_error_where, conn)
print(f"\nResultado: {len(df_error)} registros (clientes sin órdenes eliminados)")

print("\n✓ FORMA CORRECTA - Usar AND en la condición ON:\n")
query_correcta_where = '''
SELECT c.nombre, o.fecha_orden, o.monto
FROM clientes c
LEFT JOIN ordenes o ON c.id_cliente = o.id_cliente AND o.monto > 200
'''
print("Query (CORRECTA):")
print(query_correcta_where)
df_correcta_where = pd.read_sql(query_correcta_where, conn)
print(f"\nResultado: {len(df_correcta_where)} registros (mantiene todos los clientes)")
print(df_correcta_where.to_string())

## SLIDE 13: Comparación INNER JOIN vs LEFT JOIN

In [ ]:
# Comparación lado a lado
print("COMPARACIÓN: INNER JOIN vs LEFT JOIN\n")
print("="*70)

comparison_data = {
    'Característica': [
        '¿Incluye no coincidencias?',
        'Tamaño del resultado',
        'Valores NULL',
        'Caso de uso ideal',
        'En nuestro ejemplo'
    ],
    'INNER JOIN': [
        'No (solo registros con match)',
        'Igual o menor a la tabla más pequeña',
        'No hay valores NULL por el join',
        'Cuando necesitas solo datos relacionados',
        f'{len(df_inner)} registros (sin Pedro Sánchez)'
    ],
    'LEFT JOIN': [
        'Sí (incluye no coincidencias)',
        'Igual o mayor que la tabla izquierda',
        'Sí, en columnas de la tabla derecha',
        'Cuando necesitas todos de la tabla principal',
        f'{len(df_left)} registros (incluye Pedro Sánchez)'
    ]
}

df_comparison = pd.DataFrame(comparison_data)
print(df_comparison.to_string(index=False))
print("="*70)

## SLIDE 14-16: Actividad Práctica 1 - Integrando información de clientes y órdenes

**Contexto:**
Formas parte del equipo de datos de una empresa de servicios que desea evaluar el comportamiento de sus clientes. Tienes dos fuentes de información:
- Una tabla de CLIENTES
- Una tabla de ORDENES de servicio

**Objetivo:** Aplicar combinaciones INNER JOIN y LEFT JOIN sobre este modelo de datos para integrar información dispersa entre clientes y órdenes.

### SLIDE 17-18: Código Base y Consultas A Ejecutar

In [ ]:
print("ACTIVIDAD 1: Integrando información de clientes y órdenes")
print("="*70)
print("\n📋 Las tablas ya están creadas. Ejecutaremos las siguientes consultas:\n")

# CONSULTA A: INNER JOIN
print("\n✓ CONSULTA A: INNER JOIN")
print("-" * 70)
print("Objetivo: Obtener nombre de clientes y sus órdenes (solo coincidencias)")
print()

consulta_a = '''
SELECT 
    c.nombre,
    o.fecha_orden,
    o.monto,
    o.estado
FROM clientes c
INNER JOIN ordenes o ON c.id_cliente = o.id_cliente
ORDER BY c.nombre, o.fecha_orden
'''

print("SQL:")
print(consulta_a)
print("Resultado:")
df_a = pd.read_sql(consulta_a, conn)
print(df_a.to_string())
print(f"\n📊 Total registros: {len(df_a)}")

In [ ]:
# CONSULTA B: LEFT JOIN
print("\n✓ CONSULTA B: LEFT JOIN")
print("-" * 70)
print("Objetivo: Obtener todos los clientes y sus órdenes (si existen)")
print()

consulta_b = '''
SELECT 
    c.nombre,
    COALESCE(o.fecha_orden, 'Sin órdenes') as fecha_orden,
    COALESCE(CAST(o.monto AS TEXT), '-') as monto,
    COALESCE(o.estado, '-') as estado
FROM clientes c
LEFT JOIN ordenes o ON c.id_cliente = o.id_cliente
ORDER BY c.nombre, o.fecha_orden
'''

print("SQL:")
print(consulta_b)
print("Resultado:")
df_b = pd.read_sql(consulta_b, conn)
print(df_b.to_string())
print(f"\n📊 Total registros: {len(df_b)}")
print("\n💡 Nota: Observe cómo aparecen clientes sin órdenes (valores '-')")

In [ ]:
# CONSULTA ADICIONAL: Análisis de resumen
print("\n✓ CONSULTA ADICIONAL: Resumen por Cliente")
print("-" * 70)
print("Objetivo: Obtener estadísticas de órdenes por cliente usando LEFT JOIN")
print()

consulta_resumen = '''
SELECT 
    c.nombre,
    c.ciudad,
    COUNT(o.id_orden) as total_ordenes,
    COALESCE(ROUND(SUM(o.monto), 2), 0) as monto_total,
    COALESCE(ROUND(AVG(o.monto), 2), 0) as monto_promedio
FROM clientes c
LEFT JOIN ordenes o ON c.id_cliente = o.id_cliente
GROUP BY c.id_cliente, c.nombre, c.ciudad
ORDER BY total_ordenes DESC, monto_total DESC
'''

print("SQL:")
print(consulta_resumen)
print("Resultado:")
df_resumen = pd.read_sql(consulta_resumen, conn)
print(df_resumen.to_string())
print("\n💡 Esto permite identificar clientes más valiosos y aquellos sin actividad")

## SLIDE 19-23: Actividad Práctica 2 - Uniendo tablas para informes de clientes

**Contexto:**
Eres analista de datos en una empresa de soporte técnico que desea optimizar sus campañas de retención de clientes. Necesitas generar informes que muestren el comportamiento de cada cliente.

**Objetivo:** Aplicar combinaciones INNER JOIN y LEFT JOIN de manera autónoma para integrar información desde múltiples tablas relacionadas, con el fin de construir reportes completos.

In [ ]:
print("\nACTIVIDAD 2: Uniendo tablas para informes de clientes")
print("="*70)
print()

# Crear tabla adicional: TICKETS
cursor.execute('''
CREATE TABLE tickets_soporte (
    id_ticket INTEGER PRIMARY KEY,
    id_cliente INTEGER NOT NULL,
    fecha_creacion DATE NOT NULL,
    estado TEXT,
    prioridad TEXT,
    FOREIGN KEY(id_cliente) REFERENCES clientes(id_cliente)
)
''')

# Insertar datos de tickets
tickets_data = [
    (1, 1, '2024-02-10', 'cerrado', 'alta'),
    (2, 1, '2024-03-05', 'cerrado', 'media'),
    (3, 2, '2024-02-20', 'abierto', 'alta'),
    (4, 3, '2024-03-08', 'cerrado', 'baja'),
    (5, 4, '2024-03-12', 'abierto', 'media'),
    (6, 4, '2024-03-15', 'cerrado', 'alta'),
    (7, 4, '2024-03-18', 'abierto', 'media')
]

cursor.executemany(
    'INSERT INTO tickets_soporte VALUES (?, ?, ?, ?, ?)',
    tickets_data
)

conn.commit()
print("✓ Tabla 'tickets_soporte' creada con 7 tickets")
print("✓ Nota: Cliente 5 (Pedro Sánchez) NO tiene tickets de soporte\n")

df_tickets = pd.read_sql('SELECT * FROM tickets_soporte', conn)
print("Contenido de tickets_soporte:")
print(df_tickets)

In [ ]:
# CONSULTA 1: INNER JOIN - Solo clientes con tickets
print("\n✓ CONSULTA 1: INNER JOIN")
print("-" * 70)
print("Objetivo: Obtener clientes que han creado tickets de soporte\n")

consulta1 = '''
SELECT 
    c.nombre,
    c.email,
    t.id_ticket,
    t.fecha_creacion,
    t.estado,
    t.prioridad
FROM clientes c
INNER JOIN tickets_soporte t ON c.id_cliente = t.id_cliente
ORDER BY c.nombre, t.fecha_creacion
'''

print("SQL:")
print(consulta1)
print("\nResultado:")
df_query1 = pd.read_sql(consulta1, conn)
print(df_query1.to_string())
print(f"\n📊 Total registros: {len(df_query1)} (solo clientes con tickets)")

In [ ]:
# CONSULTA 2: LEFT JOIN - Todos los clientes con sus tickets
print("\n✓ CONSULTA 2: LEFT JOIN")
print("-" * 70)
print("Objetivo: Obtener todos los clientes, mostrando tickets si existen\n")

consulta2 = '''
SELECT 
    c.nombre,
    c.email,
    COALESCE(t.id_ticket, 'N/A') as id_ticket,
    COALESCE(t.fecha_creacion, 'Sin tickets') as fecha_creacion,
    COALESCE(t.estado, '-') as estado,
    COALESCE(t.prioridad, '-') as prioridad
FROM clientes c
LEFT JOIN tickets_soporte t ON c.id_cliente = t.id_cliente
ORDER BY c.nombre, t.fecha_creacion
'''

print("SQL:")
print(consulta2)
print("\nResultado:")
df_query2 = pd.read_sql(consulta2, conn)
print(df_query2.to_string())
print(f"\n📊 Total registros: {len(df_query2)} (incluye clientes sin tickets)")

In [ ]:
# CONSULTA 3: Análisis de clientes para retención
print("\n✓ CONSULTA 3: Análisis de Retención")
print("-" * 70)
print("Objetivo: Crear un informe completo para campaña de retención\n")

consulta3 = '''
SELECT 
    c.nombre,
    c.ciudad,
    COUNT(DISTINCT o.id_orden) as total_ordenes,
    COUNT(DISTINCT t.id_ticket) as total_tickets,
    COUNT(CASE WHEN t.estado = 'abierto' THEN 1 END) as tickets_abiertos,
    ROUND(SUM(o.monto), 2) as valor_total_ordenes
FROM clientes c
LEFT JOIN ordenes o ON c.id_cliente = o.id_cliente
LEFT JOIN tickets_soporte t ON c.id_cliente = t.id_cliente
GROUP BY c.id_cliente, c.nombre, c.ciudad
ORDER BY total_ordenes DESC, valor_total_ordenes DESC
'''

print("SQL:")
print(consulta3)
print("\nResultado:")
df_query3 = pd.read_sql(consulta3, conn)
print(df_query3.to_string())
print("\n💡 Este informe muestra: clientes activos, problemas técnicos y valor de negocio")

## SLIDE 24: Resumen de Conceptos

In [ ]:
print("\n" + "="*70)
print("RESUMEN DE LA SESIÓN")
print("="*70)

summary = """
✓ INNER JOIN:
  - Combina registros que TIENEN correspondencia en ambas tablas
  - Excluye registros sin coincidencia
  - Resultado ≤ tabla más pequeña
  - Uso: Cuando solo necesitas datos relacionados

✓ LEFT JOIN:
  - Mantiene TODOS los registros de la tabla izquierda
  - Incluye registros sin coincidencia (con NULL)
  - Resultado ≥ tabla izquierda
  - Uso: Cuando necesitas cobertura completa de la tabla principal

✓ BUENAS PRÁCTICAS:
  - Usa alias (c, o, t) para claridad
  - Especifica columnas explícitamente en SELECT
  - Identifica claramente las claves de relación
  - Maneja NULLs con COALESCE() en LEFT JOIN
  - Verifica la cantidad de NULL en resultados

✓ ERRORES A EVITAR:
  - Olvidar la condición ON
  - Confundir tabla izquierda/derecha
  - Agregar WHERE que convierte LEFT en INNER
  - No usar alias cuando hay nombres ambiguos
"""

print(summary)
print("="*70)

## SLIDE 26: Preguntas de Cierre para Reflexión

In [ ]:
print("\n❓ PREGUNTAS DE CIERRE PARA REFLEXIÓN:\n")
print("1. ¿Cuál es la diferencia fundamental entre INNER JOIN y LEFT JOIN?")
print("   💭 Respuesta: INNER solo incluye coincidencias, LEFT incluye todo de la tabla izquierda\n")

print("2. ¿En qué casos utilizarías un LEFT JOIN sobre un INNER JOIN?")
print("   💭 Respuesta: Cuando necesitas detectar registros huérfanos o sin correspondencia\n")

print("3. ¿Qué significa cuando aparecen valores NULL en un LEFT JOIN?")
print("   💭 Respuesta: Indica que no hay registro coincidente en la tabla derecha\n")

print("4. ¿Por qué es importante usar alias en las tablas?")
print("   💭 Respuesta: Mejora la legibilidad y evita ambigüedad con nombres de columnas\n")

print("5. ¿Cuál es el riesgo de agregar WHERE después de LEFT JOIN?")
print("   💭 Respuesta: Puede eliminar registros NULL, convirtiendo LEFT en INNER JOIN\n")

## Cierre de la Conexión

In [ ]:
# Cerrar conexión
conn.close()
print("✓ Conexión a la base de datos cerrada")
print("\n¡Fin de la Sesión 2!")

## Referencias Bibliográficas

- Coronel, C., & Morris, S. (2022). Database Systems: Design, Implementation, & Management (13th ed.). Cengage Learning.
- Silberschatz, A., Korth, H. F., & Sudarshan, S. (2019). Database System Concepts (7th ed.). McGraw-Hill.
- W3Schools SQL Tutorial: https://www.w3schools.com/sql/